# 19 Data_Collection_And_Crawling

## Problem

预训练和对齐数据最开始从哪里来？网页、代码、论文、论坛、文档这类原始语料，是如何被抓取、抽取和组织成可用数据集的？

## Dependencies

建议先理解 `14_pretraining_data_and_objective.ipynb` 和 `18_multi_stage_training_rlhf_ppo_grpo.ipynb`，这样更容易把数据采集放回完整训练流程里。


## Module_Goals

- 理解常见语料来源及其差异
- 理解 source registry、爬取、正文抽取、元数据保存的基本流程
- 理解 robots、频控、重复抓取、模板页污染这些实际问题
- 能写一个最小的数据采集管线示意

## Scope_And_Approach

这个 notebook 不追求做一个完整爬虫框架，而是把最关键的数据入口说明白：数据从哪里来、抓到的原始内容长什么样、为什么抓取策略会直接影响后面的清洗成本和训练质量。


## Context_And_Motivation

一个现代 LLM 的能力很大程度取决于语料覆盖面，但“数据多”不等于“数据好”。在真正进入清洗、过滤和训练之前，首先要面对的是原始数据采集问题。

常见来源包括：

- 网页正文：新闻、博客、文档、百科、教程
- 代码仓库：源代码、README、issue、commit message
- 论文与技术文档：PDF、HTML、LaTeX 派生文本
- 问答与论坛：高质量回答、讨论串、注释说明
- 对齐数据：人工标注问答、偏好对、评测反馈

不同来源的共同问题是：原始格式非常脏，正文和导航、广告、脚本、模板、重复页经常混在一起。


In [ ]:
source_registry = [
    {'source': 'docs_sites', 'type': 'web', 'priority': 'high', 'budget': 2000},
    {'source': 'code_repos', 'type': 'git', 'priority': 'high', 'budget': 800},
    {'source': 'papers', 'type': 'pdf_html', 'priority': 'medium', 'budget': 500},
    {'source': 'forums', 'type': 'discussion', 'priority': 'medium', 'budget': 300},
]

for item in source_registry:
    print(item)


In [ ]:
crawl_pipeline = [
    'seed_urls_or_source_registry',
    'fetch_raw_content',
    'respect_robots_and_rate_limits',
    'extract_main_text_and_metadata',
    'bucket_raw_records_by_source',
    'schedule_revisit_or_dedup',
]

for step_id, step in enumerate(crawl_pipeline, start=1):
    print(f'{step_id}. {step}')


In [ ]:
raw_record = {
    'url': 'https://example.com/article',
    'source_type': 'web_article',
    'fetch_time': '2026-05-26T22:00:00Z',
    'title': 'Example Article',
    'html_length': 128734,
    'status_code': 200,
    'language_hint': 'en',
    'main_text': 'This is the extracted body text...',
    'hash': 'sha256:example',
    'bucket': 'raw_web_high_priority',
}

for key, value in raw_record.items():
    print(f'{key}: {value}')


In [ ]:
# 一个最小正文抽取规则示意
page_segments = {
    'title': 'Example Article',
    'nav': 'Home Docs Blog About',
    'main': 'This is the extracted body text with actual content.',
    'footer': 'Privacy Terms Contact',
}

kept_fields = ['title', 'main']
extracted_text = ' '.join(page_segments[field] for field in kept_fields)
print('extracted_text =', extracted_text)


## Implementation_Notes

一个更像真实工程的数据采集系统，通常至少要解决下面这些问题：

- **Source_Registry**：先管理来源，而不是上来就散抓。不同来源有不同预算、优先级和抽取策略。
- **Frontier_Management**：决定下一个抓哪个 URL、哪个 repo、哪个文档源。
- **Fetch_Policy**：控制请求频率、重试次数、失败回退和抓取窗口。
- **Main_Content_Extraction**：把正文和导航、广告、评论区、脚本尽量拆开。
- **Metadata_Preservation**：URL、抓取时间、语言、类型、状态码、哈希值通常都必须保留。
- **Bucketization**：原始记录通常不会直接进训练，而是先进入按来源和质量分桶的数据湖。

## Trade_Offs_And_Risk_Points

- 以为爬到 HTML 就等于拿到了训练文本。真正可用的是经过抽取后的正文和元数据。
- 忽略模板页和镜像页，会让后面的去重和过滤压力暴涨。
- 只追求规模，不做来源分层，容易把低质量语料掺进高价值语料里。
- 不保存抓取元数据，后面很难追踪问题来源和做回溯清洗。
- 不做抓取预算与优先级管理，系统容易把资源浪费在低价值源上。

## Checkpoints

- 说清楚网页、代码、论文三类数据在采集时最不同的地方是什么。
- 解释为什么“正文抽取”是单独的关键步骤。
- 说明为什么原始抓取记录里要保存 URL、抓取时间、语言提示和哈希值。
- 解释为什么 source registry 和 bucketization 是数据工程起点之一。

## Summary

数据爬取解决的是“原材料从哪里来”。真正好的训练语料不是凭空出现的，而是先从多源数据里抓出来，再通过抽取、分桶、清洗、过滤和去重逐步收敛成可训练文本。采集策略本身就是模型数据质量的一部分。

## Next_Module

下一步进入数据清洗、过滤和去重。也就是把原始抓取记录真正变成可以进入 tokenizer 和训练管线的数据资产。
